# Domain F: Recurrent Neural Network Modeling

This notebook addresses two questions:
- **F1:** Can a task-trained RNN (Dale's law constrained, E/I ratio matching data) reproduce cell-type-specific tuning and connectivity?
- **F2:** Can a data-driven RNN trained to predict population ΔF/F learn biologically meaningful representations?

**Dependencies:** `torch` (PyTorch). Install via `pip install torch`.

**Note:** Both models use the actual stimulus parameters from the dataset but are trained on synthetic or simplified tasks. Sections requiring additional data or long training are marked.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
    print(f"PyTorch version: {torch.__version__}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️ PyTorch not installed. Install via: pip install torch")

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import linear_CKA
from functions.models import DalesRNN, PredictiveRNN, TemporalRNN

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

# ── Data-derived E/I ratio ──
sc_counts = obs['subclass_name'].value_counts()
exc_count = sum(sc_counts.get(s, 0) for s in present_subclasses if 'Glut' in s)
inh_count = sum(sc_counts.get(s, 0) for s in present_subclasses if 'Gaba' in s)
print(f"\nExcitatory: {exc_count} ({exc_count/len(obs):.1%}), Inhibitory: {inh_count} ({inh_count/len(obs):.1%})")
print(f"E/I ratio: {exc_count/max(inh_count,1):.1f}:1")

## F1: Task-Trained RNN with Dale's Law

Build an RNN with separate excitatory and inhibitory units (Dale's law), matching the observed E/I ratio. Train on orientation classification. Then analyze the trained network's tuning curves, noise correlations, and response to unit-type ablation.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F1.1  Dale's-Law RNN architecture (imported from functions.models)
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    print("DalesRNN model imported from functions.models.")
    print(f"Architecture: {200} exc + {20+10+10+5} inh = {245} units → 8 orientations")
else:
    print("⚠️ Skipping model definition — PyTorch not available.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F1.2  Prepare stimulus inputs and train the RNN
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    # Build stimulus dataset from actual experimental parameters
    stim_ori = var['orientation'].values.astype(float)
    stim_contrast = var['contrast'].values.astype(float)
    stim_tf = var['temporal_frequency'].values.astype(float)
    stim_running = var['avg_running'].values.astype(float)
    
    # Encode orientation as sin/cos (circular), normalize others
    ori_rad = np.deg2rad(stim_ori)
    input_features = np.column_stack([
        np.cos(ori_rad),            # orientation component 1
        np.sin(ori_rad),            # orientation component 2  
        stim_contrast / 0.8,        # normalized contrast
        np.log2(stim_tf) / 4,       # log-normalized TF
    ])
    
    # Labels: 8-way orientation classification
    ori_to_label = {o: i for i, o in enumerate(orientations)}
    labels = np.array([ori_to_label[o] for o in stim_ori])
    
    # Convert to tensors
    X_stim = torch.FloatTensor(input_features).to(device)
    y_labels = torch.LongTensor(labels).to(device)
    
    # Train/test split
    n_trials = len(labels)
    np.random.seed(42)
    perm = np.random.permutation(n_trials)
    train_idx = perm[:int(0.8 * n_trials)]
    test_idx = perm[int(0.8 * n_trials):]
    
    # Initialize model
    model = DalesRNN(n_exc=200, n_pvalb=20, n_sst=10, n_vip=10, n_lamp5=5).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    
    # Training loop
    n_epochs = 100
    batch_size = 256
    losses = []
    
    print("Training task-trained RNN...")
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        np.random.shuffle(train_idx)
        
        for start in range(0, len(train_idx), batch_size):
            batch_idx = train_idx[start:start+batch_size]
            x_batch = X_stim[batch_idx]
            y_batch = y_labels[batch_idx]
            
            outputs, rates = model(x_batch, n_steps=20)
            # Use output at last time step
            logits = outputs[:, -1, :]
            
            loss = criterion(logits, y_batch)
            # Add L2 rate penalty for biological plausibility
            rate_penalty = 1e-4 * rates.pow(2).mean()
            total_loss = loss + rate_penalty
            
            optimizer.zero_grad()
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / max(1, len(train_idx) // batch_size))
        
        if (epoch + 1) % 20 == 0:
            # Test accuracy
            model.eval()
            with torch.no_grad():
                outputs_test, _ = model(X_stim[test_idx], n_steps=20)
                logits_test = outputs_test[:, -1, :]
                preds = logits_test.argmax(dim=1)
                acc = (preds == y_labels[test_idx]).float().mean().item()
            print(f"  Epoch {epoch+1}/{n_epochs}: loss={losses[-1]:.4f}, test_acc={acc:.3f}")
    
    # ── Training curve ──
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses, 'b-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('F1: Training Loss — Task-Trained RNN', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Final test accuracy: {acc:.3f}")
else:
    print("⚠️ Skipping training — PyTorch not available.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F1.3  Analyze trained RNN: tuning curves, noise correlations, ablation
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    model.eval()
    
    # ── Extract unit tuning curves ──
    # Present each unique orientation (at contrast=0.8, TF=1) and measure firing rates
    unit_ori_tuning = np.zeros((model.n_total, len(orientations)))
    
    for oi, ori in enumerate(orientations):
        ori_rad = np.deg2rad(ori)
        stim_input = torch.FloatTensor([[np.cos(ori_rad), np.sin(ori_rad), 1.0, 0.0]]).to(device)
        # Run multiple times for noise averaging
        rates_samples = []
        for _ in range(50):
            model.train()  # enable noise
            with torch.no_grad():
                _, rates = model(stim_input, n_steps=20)
                rates_samples.append(rates[0, -1, :].cpu().numpy())
        model.eval()
        unit_ori_tuning[:, oi] = np.mean(rates_samples, axis=0)
    
    # ── Plot tuning curves by unit type ──
    unit_type_arr = np.array(model.unit_types)
    unique_types = ['L2/3 IT', 'L4/5 IT', 'Pvalb', 'Sst', 'Vip', 'Lamp5']
    type_colors = {'L2/3 IT': '#1f77b4', 'L4/5 IT': '#2ca02c', 'Pvalb': '#d62728',
                   'Sst': '#ff7f0e', 'Vip': '#e377c2', 'Lamp5': '#8c564b'}
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, utype in zip(axes.flat, unique_types):
        type_mask = unit_type_arr == utype
        n_units = type_mask.sum()
        if n_units == 0:
            ax.set_visible(False)
            continue
        
        tuning = unit_ori_tuning[type_mask]
        mean_t = np.mean(tuning, axis=0)
        sem_t = np.std(tuning, axis=0) / np.sqrt(n_units)
        
        ax.errorbar(orientations, mean_t, yerr=sem_t, color=type_colors[utype],
                    linewidth=2, capsize=3, marker='o')
        # Show individual units
        for i in range(min(20, n_units)):
            ax.plot(orientations, tuning[i], color=type_colors[utype], alpha=0.15, linewidth=0.5)
        
        ax.set_title(f'{utype} (n={n_units})', color=type_colors[utype], fontweight='bold')
        ax.set_xlabel('Direction (°)')
        ax.set_ylabel('Firing Rate')
        ax.set_xticks(orientations)
    
    plt.suptitle('F1: RNN Unit Tuning Curves by Type', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # ── Effective weight matrix visualization ──
    W_eff = model.get_effective_W().detach().cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    ax = axes[0]
    im = ax.imshow(W_eff, aspect='auto', cmap='RdBu_r', vmin=-np.percentile(np.abs(W_eff), 95),
                   vmax=np.percentile(np.abs(W_eff), 95))
    plt.colorbar(im, ax=ax)
    # Mark E/I boundary
    ax.axhline(model.n_exc - 0.5, color='yellow', lw=1)
    ax.axvline(model.n_exc - 0.5, color='yellow', lw=1)
    ax.set_title('F1: Effective Recurrent Weights', fontweight='bold')
    ax.set_xlabel('From unit')
    ax.set_ylabel('To unit')
    
    # E→E, E→I, I→E, I→I block means
    ax = axes[1]
    ne = model.n_exc
    block_means = {
        'E→E': np.mean(W_eff[:ne, :ne]),
        'E→I': np.mean(W_eff[ne:, :ne]),
        'I→E': np.mean(W_eff[:ne, ne:]),
        'I→I': np.mean(W_eff[ne:, ne:]),
    }
    ax.bar(block_means.keys(), block_means.values(),
           color=['green', 'orange', 'red', 'purple'])
    ax.set_ylabel('Mean Weight')
    ax.set_title('F1: Mean Connection Weights by Block', fontweight='bold')
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()
    
    # ── Ablation experiment ──
    print("\n=== Ablation Experiment ===")
    # Baseline accuracy
    with torch.no_grad():
        outputs_base, _ = model(X_stim[test_idx], n_steps=20)
        base_acc = (outputs_base[:, -1, :].argmax(1) == y_labels[test_idx]).float().mean().item()
    print(f"Baseline accuracy: {base_acc:.3f}")
    
    ablation_results = {}
    for abl_type in unique_types:
        # Zero out firing rates of this unit type
        type_mask_t = torch.tensor([t == abl_type for t in model.unit_types], dtype=torch.bool)
        
        # Temporarily modify forward pass by masking rates
        saved_bias = model.bias.data.clone()
        # Set bias to large negative for ablated units (effectively silencing them)
        model.bias.data[type_mask_t] = -100.0
        
        with torch.no_grad():
            outputs_abl, _ = model(X_stim[test_idx], n_steps=20)
            abl_acc = (outputs_abl[:, -1, :].argmax(1) == y_labels[test_idx]).float().mean().item()
        
        model.bias.data = saved_bias  # restore
        drop = base_acc - abl_acc
        ablation_results[abl_type] = {'acc': abl_acc, 'acc_drop': drop}
        print(f"  Ablate {abl_type:10s}: acc={abl_acc:.3f}, drop={drop:+.3f}")
    
    # ── Ablation bar chart ──
    fig, ax = plt.subplots(figsize=(10, 5))
    types_list = list(ablation_results.keys())
    drops = [ablation_results[t]['acc_drop'] for t in types_list]
    colors_abl = [type_colors.get(t, 'gray') for t in types_list]
    ax.bar(types_list, drops, color=colors_abl)
    ax.set_ylabel('Accuracy Drop')
    ax.set_title('F1: Ablation Impact on Orientation Decoding', fontweight='bold')
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping analysis — PyTorch not available.")